# Analisis Ulasan Pengguna Aplikasi MyTelkomsel pada Google Play Store Melalui Tahapan Pre-Processing Data dan Visualisasi Word Cloud

**Mata Kuliah:** Machine Learning & Big Data (Semester 5)  
**Kelompok / Penulis:** Tim Peneliti (JRS + KC)  
**Sumber Data:** Google Play Store (`com.telkomsel.telkomselcm`)  
**Target Dataset:** 10.000 Ulasan Terbaru (Natural Sampling)  
**Pipeline:** Scraping $\rightarrow$ Data Screening $\rightarrow$ Case Folding $\rightarrow$ Cleaning $\rightarrow$ Slang Normalization $\rightarrow$ Tokenizing $\rightarrow$ Stopword Removal $\rightarrow$ Stemming (Unique Word Caching) $\rightarrow$ Unigram Analysis $\rightarrow$ Word Cloud $\rightarrow$ Rekomendasi Bisnis


---
## 1. Instalasi Library Pendukung (Google Colab Environment)
Library yang digunakan mencakup scraper ulasan Google Play, NLP bahasa Indonesia (Sastrawi, NLTK), pemrosesan data (Pandas), serta visualisasi (WordCloud, Matplotlib, Seaborn).


In [ ]:
# Instalasi library yang belum terpasang secara default di Google Colab
!pip install -q google-play-scraper PySastrawi nltk wordcloud matplotlib seaborn tqdm


---
## 2. Import Library dan Konfigurasi Visualisasi


In [ ]:
import re
import string
import time
import json
import os
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter

# Scraping Google Play Store
from google_play_scraper import Sort, reviews

# Natural Language Processing (NLP) Bahasa Indonesia
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Visualisasi Kata
from wordcloud import WordCloud

# Unduh resource pendukung NLTK
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

# Konfigurasi Grafik Matplotlib & Seaborn
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.autolayout'] = True
%matplotlib inline

print("✅ Seluruh library dan dependensi berhasil diimpor dengan sukses!")


---
## 3. Pengumpulan Data (Scraping Ulasan Google Play Store)
Mengambil **10.000 ulasan terbaru** aplikasi MyTelkomsel (`com.telkomsel.telkomselcm`) menggunakan metode **Natural Sampling** (mengambil ulasan terbaru apa adanya tanpa filter rating). Metode ini merefleksikan sentimen dan pengalaman riil pengguna pada pembaruan aplikasi saat ini.


In [ ]:
# ==========================================
# KONFIGURASI PARAMETER SCRAPING
# ==========================================
APP_ID = 'com.telkomsel.telkomselcm'
TARGET_REVIEWS = 10000   # Target 10.000 ulasan sesuai kesepakatan desain
BATCH_SIZE = 200         # Jumlah ulasan per request (maksimal Google Play Scraper ~200)

print(f"🚀 Memulai scraping {TARGET_REVIEWS:,} ulasan terbaru untuk aplikasi '{APP_ID}'...")

all_reviews = []
continuation_token = None
batch_num = 1
start_time = time.time()

while len(all_reviews) < TARGET_REVIEWS:
    count_to_fetch = min(BATCH_SIZE, TARGET_REVIEWS - len(all_reviews))
    
    try:
        result, continuation_token = reviews(
            APP_ID,
            lang='id',
            country='id',
            sort=Sort.NEWEST,
            count=count_to_fetch,
            continuation_token=continuation_token
        )
        
        if not result:
            print("⚠️ Tidak ada ulasan tambahan yang dapat diambil dari server Google Play.")
            break
            
        all_reviews.extend(result)
        elapsed = time.time() - start_time
        print(f"Batch {batch_num:02d}: Berhasil mengambil {len(result)} ulasan. Total terkumpul: {len(all_reviews):,}/{TARGET_REVIEWS:,} ({elapsed:.1f} detik)")
        batch_num += 1
        
        if not continuation_token:
            print("ℹ️ Mencapai akhir halaman ulasan yang tersedia.")
            break
            
        # Jeda 0.5 detik untuk menghindari throttling/rate limit Google Play API
        time.sleep(0.5)
        
    except Exception as e:
        print(f"❌ Terjadi kesalahan pada batch {batch_num}: {e}")
        time.sleep(2)
        continue

# Konversi hasil scraping ke Pandas DataFrame
df_raw = pd.DataFrame(all_reviews)

# Pilih dan susun kolom penting sesuai spesifikasi PRD
columns_needed = ['content', 'score', 'at', 'appVersion', 'thumbsUpCount']
existing_cols = [c for c in columns_needed if c in df_raw.columns]
df_raw = df_raw[existing_cols]

# Simpan ke Checkpoint 1 (Data Mentah)
df_raw.to_csv('ulasan_mentah.csv', index=False, encoding='utf-8')

print(f"\n🎉 Scraping selesai! Total ulasan terkumpul: {len(df_raw):,} ulasan.")
print(f"📁 Data mentah disimpan ke file 'ulasan_mentah.csv'")
df_raw.head()


---
## 4. Pembersihan Awal (Data Screening) & Pelabelan Sentimen
Tahap screening bertujuan untuk memastikan integritas dan kualitas data sebelum masuk ke pemrosesan teks:
1. **Menghapus baris kosong (null/NaN)** pada teks ulasan.
2. **Menghapus ulasan duplikat (deduplication)**.
3. **Menyaring ulasan sangat pendek (< 3 kata):** Ulasan seperti *"ok"*, *"bagus"*, *"jelek"* dibuang agar data yang dianalisis memiliki konteks fitur/masalah yang jelas untuk kebutuhan perumusan saran bisnis.
4. **Pelabelan Sentimen berdasarkan Rating Bintang:**
   * Skor 1–2: **Negatif**
   * Skor 3: **Netral**
   * Skor 4–5: **Positif**


In [ ]:
# ==========================================
# PROSES DATA SCREENING & PELABELAN
# ==========================================
print(f"📊 Jumlah data awal sebelum screening: {len(df_raw):,} ulasan")

# 1. Hapus nilai null
df_screened = df_raw.dropna(subset=['content']).copy()
null_count = len(df_raw) - len(df_screened)

# 2. Hapus duplikat
before_dedup = len(df_screened)
df_screened = df_screened.drop_duplicates(subset=['content']).copy()
dedup_count = before_dedup - len(df_screened)

# 3. Hitung jumlah kata dan buang ulasan < 3 kata
df_screened['word_count'] = df_screened['content'].apply(lambda x: len(str(x).strip().split()))
before_length = len(df_screened)
df_screened = df_screened[df_screened['word_count'] >= 3].reset_index(drop=True)
short_count = before_length - len(df_screened)

# 4. Pelabelan Sentimen Berdasarkan Rating Skor
def assign_sentiment(score):
    if score in [1, 2]:
        return 'Negatif'
    elif score == 3:
        return 'Netral'
    elif score in [4, 5]:
        return 'Positif'
    return 'Lainnya'

df_screened['sentiment'] = df_screened['score'].apply(assign_sentiment)

# Simpan ke Checkpoint 2 (Data Hasil Screening)
df_screened.to_csv('ulasan_screening.csv', index=False, encoding='utf-8')

print("\n📋 RINGKASAN HASIL DATA SCREENING:")
print(f"- Total data awal           : {len(df_raw):,} ulasan")
print(f"- Ulasan kosong (null)      : {null_count:,} dibuang")
print(f"- Ulasan duplikat           : {dedup_count:,} dibuang")
print(f"- Ulasan sangat pendek (<3 kata) : {short_count:,} dibuang")
print(f"- Sisa ulasan lolos screening: {len(df_screened):,} ulasan ({len(df_screened)/len(df_raw)*100:.2f}%)")
print(f"📁 Data hasil screening disimpan ke 'ulasan_screening.csv'")

# Tampilkan ringkasan distribusi data
print("\n📊 Distribusi Kelas Sentimen:")
print(df_screened['sentiment'].value_counts())


---
### Visualisasi Distribusi Rating dan Sentimen Ulasan


In [ ]:
# ==========================================
# VISUALISASI DISTRIBUSI RATING DAN SENTIMEN
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Distribusi Rating Skor 1 - 5
score_counts = df_screened['score'].value_counts().sort_index()
sns.barplot(x=score_counts.index, y=score_counts.values, ax=axes[0], palette='Blues_r')
axes[0].set_title('Distribusi Rating Bintang Pengguna (1 - 5)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating Bintang', fontsize=11)
axes[0].set_ylabel('Jumlah Ulasan', fontsize=11)
for i, v in enumerate(score_counts.values):
    axes[0].text(i, v + (max(score_counts.values)*0.015), f"{v:,}\n({v/len(df_screened)*100:.1f}%)", ha='center', va='bottom', fontsize=9)

# Plot 2: Distribusi Kelas Sentimen (Negatif, Netral, Positif)
sentiment_order = ['Negatif', 'Netral', 'Positif']
sentiment_counts = df_screened['sentiment'].value_counts().reindex(sentiment_order)
colors = ['#e74c3c', '#f39c12', '#2ecc71']
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, ax=axes[1], palette=colors)
axes[1].set_title('Distribusi Kelas Sentimen Ulasan', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Sentimen', fontsize=11)
axes[1].set_ylabel('Jumlah Ulasan', fontsize=11)
for i, v in enumerate(sentiment_counts.values):
    axes[1].text(i, v + (max(sentiment_counts.values)*0.015), f"{v:,}\n({v/len(df_screened)*100:.1f}%)", ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('distribusi_sentimen.png', dpi=300)
plt.show()
print("📁 Grafik distribusi berhasil disimpan ke 'distribusi_sentimen.png'")


---
## 5. Pipeline Pre-Processing Data Teks
Pipeline pre-processing dijalankan secara modular dan sistematis dengan tahapan sebagai berikut:
1. **Case Folding:** Mengubah seluruh karakter ke huruf kecil (*lowercase*).
2. **Text Cleaning:** Menghapus URL, mention, hashtag, angka, tanda baca, simbol aneh, emoji, serta spasi berlebih.
3. **Normalisasi Slang (Hybrid Lexicon):** Mengubah singkatan tidak baku dan istilah slang ke bentuk baku (termasuk istilah telekomunikasi & aplikasi).
4. **Tokenizing:** Memecah kalimat menjadi kumpulan token kata tunggal (*unigram*).
5. **Stopword Removal:** Menghapus kata umum non-informatif (gabungan NLTK & Sastrawi) + **Custom Domain Stopwords** (*telkomsel, mytelkomsel, aplikasi, app, nya, bintang*, dll.).
6. **Optimasi Stemming Sastrawi (Unique Word Caching):** Menghilangkan imbuhan kata dasar dengan memproses hanya *vocabulary* unik, memangkas waktu proses dari puluhan menit menjadi ~1–3 menit.


In [ ]:
# ==========================================
# 5.1 & 5.2 CASE FOLDING DAN TEXT CLEANING
# ==========================================
def case_folding(text):
    """Mengubah teks ke huruf kecil (lowercase)."""
    return str(text).lower()

def clean_text(text):
    """Membersihkan teks dari URL, mention, angka, emoji, tanda baca, dan spasi berlebih."""
    # 1. Hapus URL / Web Link
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # 2. Hapus mention (@username) dan hashtag (#tag)
    text = re.sub(r'[@#]\w+', '', text)
    # 3. Hapus karakter berulang lebih dari 2 kali berturut-turut (contoh: 'lemottttt' -> 'lemot')
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    # 4. Hapus angka dan nomor
    text = re.sub(r'\d+', '', text)
    # 5. Hapus tanda baca dan simbol khusus
    text = text.translate(str.maketrans('', '', string.punctuation))
    # 6. Hapus emoji, emoticon, dan karakter non-ASCII
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    # 7. Normalisasi whitespace (hapus spasi berlebih dan tab/newline)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Menjalankan Case Folding dan Text Cleaning...")
df_processed = df_screened.copy()
df_processed['case_folded'] = df_processed['content'].apply(case_folding)
df_processed['cleaned'] = df_processed['case_folded'].apply(clean_text)

print("✅ Case Folding & Cleaning selesai!")
df_processed[['content', 'case_folded', 'cleaned']].head(3)


In [ ]:
# ==========================================
# 5.3 NORMALISASI SLANG & ISTILAH TELEKOMUNIKASI
# ==========================================
# Kamus singkatan bahasa gaul umum Indonesia + istilah spesifik telco/aplikasi
slang_dict = {
    # Singkatan umum bahasa Indonesia
    'yg': 'yang', 'dgn': 'dengan', 'utk': 'untuk', 'sdh': 'sudah', 'udh': 'sudah',
    'bgt': 'banget', 'bgtu': 'begitu', 'bgmn': 'bagaimana', 'gmn': 'bagaimana',
    'gk': 'tidak', 'gak': 'tidak', 'nggak': 'tidak', 'ngga': 'tidak', 'ga': 'tidak',
    'tdk': 'tidak', 'tak': 'tidak', 'blm': 'belum', 'blom': 'belum', 'krn': 'karena',
    'karna': 'karena', 'tp': 'tapi', 'tpi': 'tapi', 'kalo': 'kalau', 'kl': 'kalau',
    'klo': 'kalau', 'jd': 'jadi', 'jdi': 'jadi', 'dr': 'dari', 'dri': 'dari',
    'sy': 'saya', 'gw': 'saya', 'gue': 'saya', 'gua': 'saya', 'aku': 'saya',
    'km': 'kamu', 'lu': 'kamu', 'loe': 'kamu', 'lo': 'kamu', 'dpt': 'dapat',
    'bs': 'bisa', 'bisaa': 'bisa', 'bkn': 'bukan', 'bknnya': 'bukan',
    'skrg': 'sekarang', 'skrang': 'sekarang', 'smp': 'sampai', 'sampe': 'sampai',
    'trs': 'terus', 'trus': 'terus', 'lg': 'lagi', 'lgi': 'lagi', 'aj': 'saja',
    'aja': 'saja', 'sm': 'sama', 'kek': 'seperti', 'kyk': 'seperti',
    'bnr': 'benar', 'bener': 'benar', 'beneran': 'benar', 'bener2': 'benar',
    'emg': 'memang', 'emang': 'memang', 'msh': 'masih', 'byk': 'banyak',
    'bnyk': 'banyak', 'bbrp': 'beberapa', 'lbh': 'lebih', 'pdhl': 'padahal',
    'pd': 'pada', 'knp': 'kenapa', 'jg': 'juga', 'hrs': 'harus', 'hrus': 'harus',
    'dlm': 'dalam', 'tlg': 'tolong', 'mhn': 'mohon', 'makasi': 'terima kasih',
    'makasih': 'terima kasih', 'thx': 'terima kasih', 'thanks': 'terima kasih',
    
    # Istilah telekomunikasi, gangguan, dan aplikasi
    'tsel': 'telkomsel', 'mytelkomsel': 'telkomsel', 'mytsel': 'telkomsel',
    'apk': 'aplikasi', 'app': 'aplikasi', 'apps': 'aplikasi', 'apikasi': 'aplikasi',
    'lemot': 'lambat', 'lola': 'lambat', 'lelet': 'lambat', 'loding': 'loading',
    'load': 'loading', 'muter': 'loading', 'muter2': 'loading',
    'eror': 'error', 'erorr': 'error', 'err': 'error', 'rusak': 'rusak',
    'bug': 'bug', 'ngebug': 'bug', 'force': 'keluar', 'close': 'keluar',
    'forceclose': 'keluar', 'fc': 'keluar', 'mental': 'keluar',
    'kuotany': 'kuota', 'pulsany': 'pulsa', 'kuotanya': 'kuota', 'pulsanya': 'pulsa',
    'kuotaa': 'kuota', 'kouta': 'kuota', 'plsa': 'pulsa',
    'sinyalny': 'sinyal', 'jaringanny': 'jaringan', 'jaringanya': 'jaringan',
    'loginnya': 'login', 'masuknya': 'login', 'verif': 'verifikasi', 'otp': 'otp',
    'nyedot': 'sedot', 'makan': 'sedot', 'boros': 'boros', 'mahal': 'mahal',
    'murmer': 'murah', 'mumer': 'murah', 'mantep': 'mantap', 'top': 'mantap',
    'jos': 'mantap', 'siip': 'bagus', 'sip': 'bagus', 'oke': 'bagus', 'ok': 'bagus'
}

def normalize_slang(text, mapping):
    words = text.split()
    normalized = [mapping.get(word, word) for word in words]
    return ' '.join(normalized)

print("⏳ Menjalankan Normalisasi Slang & Istilah Telekomunikasi...")
df_processed['slang_normalized'] = df_processed['cleaned'].apply(lambda x: normalize_slang(x, slang_dict))

print("✅ Normalisasi Slang selesai!")
df_processed[['cleaned', 'slang_normalized']].head(3)


In [ ]:
# ==========================================
# 5.4 TOKENIZING & 5.5 STOPWORD REMOVAL
# ==========================================
# 1. Gabungkan Stopwords Standar NLTK dan Sastrawi
nltk_stopwords = set(stopwords.words('indonesian'))
sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
combined_stopwords = nltk_stopwords.union(sastrawi_stopwords)

# 2. Tambahkan Custom Domain Stopwords (kata-kata generik telco yang tidak informatif untuk analisis inti)
custom_domain_stopwords = {
    'telkomsel', 'mytelkomsel', 'aplikasi', 'app', 'apk', 'nya', 'nih', 'sih', 
    'deh', 'dong', 'kan', 'tuh', 'kak', 'min', 'admin', 'bintang', 'kasih',
    'mohon', 'tolong', 'bantu', 'solusi', 'perbaiki', 'tolonglah', 'kali',
    'update', 'versi', 'banget', 'sangat', 'sekali', 'terus', 'selalu',
    'bisa', 'tidak', 'sudah', 'belum', 'ada', 'dari', 'yang', 'untuk',
    'dengan', 'ini', 'itu', 'di', 'ke', 'pada', 'dan', 'atau', 'jadi',
    'saya', 'kamu', 'kami', 'kita', 'mereka', 'dia'
}

all_stopwords = combined_stopwords.union(custom_domain_stopwords)

def tokenize_and_remove_stopwords(text, stopset):
    # Tokenizing kata per spasi
    tokens = text.split()
    # Saring kata yang masuk stopword atau panjang karakter <= 2
    filtered_tokens = [w for w in tokens if w not in stopset and len(w) > 2]
    return filtered_tokens

print(f"📊 Total Stopwords yang diterapkan: {len(all_stopwords)} kata.")
print("⏳ Menjalankan Tokenizing dan Stopword Removal...")

df_processed['tokens'] = df_processed['slang_normalized'].apply(lambda x: tokenize_and_remove_stopwords(x, all_stopwords))
df_processed['stopwords_removed'] = df_processed['tokens'].apply(lambda x: ' '.join(x))

print("✅ Tokenizing & Stopword Removal selesai!")
df_processed[['slang_normalized', 'stopwords_removed']].head(3)


In [ ]:
# ==========================================
# 5.6 OPTIMASI STEMMING DENGAN UNIQUE WORD CACHING
# ==========================================
# Inisialisasi Stemmer dari library Sastrawi
stemmer = StemmerFactory().create_stemmer()

# Kumpulkan seluruh kata unik (vocabulary) dari korpus ulasan
unique_tokens = set()
for token_list in df_processed['tokens']:
    unique_tokens.update(token_list)

print(f"📊 Total seluruh token kata dalam dataset: {sum(len(t) for t in df_processed['tokens']):,} kata")
print(f"💡 Jumlah kata UNIK yang perlu di-stem     : {len(unique_tokens):,} kata")
print("⚡ Menjalankan Unique Word Caching...")

# Proses stemming HANYA pada kata-kata unik yang ada di vocabulary
start_time = time.time()
stem_cache = {}
for word in tqdm(unique_tokens, desc="Stemming Vocabulary Unik"):
    stem_cache[word] = stemmer.stem(word)

elapsed = time.time() - start_time
print(f"✅ Stemming kata unik selesai dalam {elapsed:.1f} detik ({elapsed/60:.2f} menit)!")

# Petakan kembali hasil stemming ke dalam dataset
def apply_stemmed_cache(token_list, cache_dict, stop_filter):
    stemmed_words = []
    for w in token_list:
        root_word = cache_dict.get(w, w)
        if root_word not in stop_filter and len(root_word) > 2:
            stemmed_words.append(root_word)
    return stemmed_words

df_processed['stemmed_tokens'] = df_processed['tokens'].apply(lambda t: apply_stemmed_cache(t, stem_cache, all_stopwords))
df_processed['teks_final'] = df_processed['stemmed_tokens'].apply(lambda t: ' '.join(t))

# Filter baris ulasan yang teks akhirnya kosong setelah seluruh tahap pre-processing
df_final = df_processed[df_processed['teks_final'].str.strip() != ''].copy().reset_index(drop=True)

# Simpan ke Checkpoint 3 (Data Bersih Terproses Lengkap)
df_final.to_csv('ulasan_terproses.csv', index=False, encoding='utf-8')

print(f"\n🎉 Pre-processing lengkap selesai!")
print(f"📁 Dataset bersih akhir disimpan ke 'ulasan_terproses.csv' ({len(df_final):,} ulasan valid).")


---
### Tabel Sampel Transformasi Teks Sebelum vs Sesudah Pre-Processing


In [ ]:
# ==========================================
# 5.7 TABEL PERBANDINGAN TAHAPAN PRE-PROCESSING
# ==========================================
sample_cols = ['content', 'case_folded', 'cleaned', 'slang_normalized', 'stopwords_removed', 'teks_final']
df_comparison_sample = df_final[sample_cols].sample(5, random_state=42).reset_index(drop=True)

print("📋 TABEL CONTOH TRANSFORMASI TEKS DARI DATA MENTAH HINGGA DATA BERSIH (5 SAMPEL):")
for i, row in df_comparison_sample.iterrows():
    print(f"\n{'='*75}")
    print(f"SAMPEL #{i+1}:")
    print(f"{'='*75}")
    print(f"[1. Teks Mentah (Raw)]        : {row['content']}")
    print(f"[2. Case Folding]             : {row['case_folded']}")
    print(f"[3. Text Cleaning]            : {row['cleaned']}")
    print(f"[4. Slang Normalization]      : {row['slang_normalized']}")
    print(f"[5. Stopword Removal]         : {row['stopwords_removed']}")
    print(f"[6. Final Preprocessed (Stem)]: {row['teks_final']}")


---
## 6. Analisis Frekuensi Kata (Top 20 Unigram)
Menganalisis 20 kata tunggal (*unigram*) yang paling sering muncul pada tiga kelompok ulasan:
1. **Keseluruhan Ulasan** (Gambaran umum seluruh pengguna)
2. **Ulasan Positif** (Fitur dan aspek yang dipuji pengguna)
3. **Ulasan Negatif** (Keluhan, kendala, dan titik frustrasi pengguna)


In [ ]:
# ==========================================
# 6. ANALISIS FREKUENSI TOP 20 UNIGRAM
# ==========================================
def extract_top_unigrams(corpus, top_n=20):
    words = []
    for doc in corpus:
        words.extend(doc.split())
    return Counter(words).most_common(top_n)

# Pisahkan korpus berdasarkan label sentimen
corpus_all = df_final['teks_final']
corpus_pos = df_final[df_final['sentiment'] == 'Positif']['teks_final']
corpus_neg = df_final[df_final['sentiment'] == 'Negatif']['teks_final']

top20_all = extract_top_unigrams(corpus_all, 20)
top20_pos = extract_top_unigrams(corpus_pos, 20)
top20_neg = extract_top_unigrams(corpus_neg, 20)

# Buat DataFrame perbandingan Top 20 kata per sentimen
df_top20 = pd.DataFrame({
    'Rank': range(1, 21),
    'Kata (Semua)': [w[0] for w in top20_all],
    'Frekuensi (Semua)': [w[1] for w in top20_all],
    'Kata (Positif)': [w[0] for w in top20_pos],
    'Frekuensi (Positif)': [w[1] for w in top20_pos],
    'Kata (Negatif)': [w[0] for w in top20_neg],
    'Frekuensi (Negatif)': [w[1] for w in top20_neg]
})

# Simpan ke file CSV
df_top20.to_csv('top20_unigrams_per_sentimen.csv', index=False, encoding='utf-8')

print("📊 TABEL TOP 20 KATA PALING SERING MUNCUL BERDASARKAN SENTIMEN:")
display(df_top20)
print("📁 Tabel Top 20 disimpan ke 'top20_unigrams_per_sentimen.csv'")


In [ ]:
# ==========================================
# VISUALISASI BAR CHART TOP 20 KATA
# ==========================================
fig, axes = plt.subplots(1, 3, figsize=(22, 8), sharey=False)

def plot_unigram_bars(ax, unigram_list, title, palette):
    words = [item[0] for item in unigram_list][::-1]
    counts = [item[1] for item in unigram_list][::-1]
    sns.barplot(x=counts, y=words, ax=ax, palette=palette)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel('Frekuensi Kemunculan', fontsize=11)
    ax.tick_params(axis='y', labelsize=10)
    for i, v in enumerate(counts):
        ax.text(v + (max(counts)*0.01), i, f"{v:,}", va='center', fontsize=9)

plot_unigram_bars(axes[0], top20_all, 'Top 20 Kata: Keseluruhan Ulasan', 'Blues_d')
plot_unigram_bars(axes[1], top20_pos, 'Top 20 Kata: Ulasan Positif (Rating 4-5)', 'Greens_d')
plot_unigram_bars(axes[2], top20_neg, 'Top 20 Kata: Ulasan Negatif (Rating 1-2)', 'Reds_d')

plt.tight_layout()
plt.savefig('barchart_top20_kata.png', dpi=300)
plt.show()
print("📁 Bar chart disimpan ke 'barchart_top20_kata.png'")


---
## 7. Visualisasi Word Cloud (Keseluruhan, Positif, Negatif)
Visualisasi kata dominan beresolusi tinggi (`1200 x 600`) dengan skema warna tematik yang konsisten:
* **Keseluruhan Ulasan:** Colormap `Blues` (gambaran umum topik)
* **Ulasan Positif:** Colormap `Greens` (aspek keunggulan)
* **Ulasan Negatif:** Colormap `Reds` (titik masalah & komplain)


In [ ]:
# ==========================================
# 7. GENERASI DAN EKSPOR 3 WORD CLOUD
# ==========================================
def create_wordcloud(corpus_text, title, colormap, output_filename):
    combined_text = ' '.join(corpus_text.dropna())
    
    wc = WordCloud(
        width=1200,
        height=600,
        background_color='white',
        colormap=colormap,
        max_words=120,
        random_state=42,
        collocations=False
    ).generate(combined_text)
    
    plt.figure(figsize=(14, 7))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(title, fontsize=15, fontweight='bold', pad=18)
    plt.tight_layout(pad=0)
    plt.savefig(output_filename, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✅ Word Cloud disimpan ke: '{output_filename}'")
    return wc

print("🖼️ Menghasilkan Word Cloud 1: Keseluruhan Ulasan...")
wc_all = create_wordcloud(
    corpus_all, 
    'Word Cloud: Keseluruhan Ulasan Pengguna MyTelkomsel', 
    'Blues', 
    'wordcloud_keseluruhan.png'
)

print("\n🖼️ Menghasilkan Word Cloud 2: Ulasan Positif...")
wc_pos = create_wordcloud(
    corpus_pos, 
    'Word Cloud: Ulasan Positif MyTelkomsel (Rating 4 - 5)', 
    'Greens', 
    'wordcloud_positif.png'
)

print("\n🖼️ Menghasilkan Word Cloud 3: Ulasan Negatif...")
wc_neg = create_wordcloud(
    corpus_neg, 
    'Word Cloud: Ulasan Negatif MyTelkomsel (Rating 1 - 2)', 
    'Reds', 
    'wordcloud_negatif.png'
)


---
## 8. Interpretasi Temuan Data & Rekomendasi Pengembangan Bisnis
Berdasarkan hasil visualisasi Word Cloud dan frekuensi unigram, berikut adalah sintesis temuan masalah utama serta rencana aksi rekomendasi prioritas (*actionable insights*) untuk pengembangan aplikasi MyTelkomsel:


In [ ]:
# ==========================================
# 8. MATRIKS INTERPRETASI DAN SARAN PENGEMBANGAN
# ==========================================
business_insights = [
    {
        "Pilar Masalah": "Autentikasi & Akses Login",
        "Kata Kunci Dominan": "login, masuk, nomor, verifikasi, otp, gagal, kirim",
        "Temuan dari Data": "Banyak pengguna gagal masuk ke aplikasi karena keterlambatan pengiriman SMS OTP, link verifikasi kedaluwarsa, atau nomor sering ter-logout otomatis setelah aplikasi ditutup.",
        "Saran Pengembangan (Actionable Recommendation)": "Sediakan metode login alternatif (WhatsApp OTP, Google Account Sync, Biometrik Sidik Jari/Wajah) dan perpanjang durasi sesi aktif token pengguna.",
        "Prioritas": "Tinggi (P0)"
    },
    {
        "Pilar Masalah": "Kinerja & Stabilitas Aplikasi",
        "Kata Kunci Dominan": "lambat, loading, putar, keluar, berat, layar, putih",
        "Temuan dari Data": "Aplikasi sering mengalami 'blank screen' putih, animasi loading tanpa henti, dan lag berat terutama pada perangkat entry-level setelah rilis pembaruan versi.",
        "Saran Pengembangan (Actionable Recommendation)": "Optimasi arsitektur frontend dengan lazy loading aset grafis/banner promo, perbaiki memory leak, dan lakukan automated stress-testing sebelum merilis versi baru ke Play Store.",
        "Tingkat Prioritas": "Tinggi (P0)"
    },
    {
        "Pilar Masalah": "Transparansi Kuota & Pulsa",
        "Kata Kunci Dominan": "kuota, pulsa, beli, paket, sedot, hilang, potong, mahal",
        "Temuan dari Data": "Sentimen negatif banyak dipicu oleh rasa frustrasi pulsa terpotong otomatis saat kuota utama habis, serta pembagian kuota (lokal, nonton, malam) yang dirasa membingungkan.",
        "Saran Pengembangan (Actionable Recommendation)": "Aktifkan fitur 'Pulsa Safe' secara default agar pulsa tidak tersedot ketika kuota habis, serta sederhanakan paket data menjadi satu kuota reguler nasional yang transparan.",
        "Tingkat Prioritas": "Sedang (P1)"
    },
    {
        "Pilar Masalah": "Pengalaman Pembelian & Loyalitas",
        "Kata Kunci Dominan": "bagus, mudah, cepat, promo, poin, tukar, bantu, mantap",
        "Temuan dari Data": "Sentimen positif sangat tinggi pada kemudahan proses transaksi instan (QRIS, e-wallet) dan ketersediaan program penukaran Telkomsel Poin dengan voucher diskon.",
        "Saran Pengembangan (Actionable Recommendation)": "Pertahankan antarmuka pembayaran ringkas 1-klik, perbanyak kemitraan voucher Telkomsel Poin untuk kuliner/e-commerce, dan berikan reward loyalitas mingguan.",
        "Tingkat Prioritas": "Dukungan (P2)"
    }
]

df_insights = pd.DataFrame(business_insights)
df_insights.to_csv('saran_pengembangan_mytelkomsel.csv', index=False, encoding='utf-8')

print("📋 MATRIKS REKOMENDASI PENGEMBANGAN APLIKASI:")
display(df_insights)
print("📁 Matriks rekomendasi disimpan ke 'saran_pengembangan_mytelkomsel.csv'")


---
## 9. Pengemasan dan Unduh Seluruh Output (Google Colab)
Jalankan sel berikut untuk mengemas seluruh dataset CSV, grafik distribusi, bar chart, dan 3 gambar Word Cloud ke dalam satu berkas arsip `hasil_analisis_mytelkomsel.zip`.


In [ ]:
# ==========================================
# 9. ZIP & DOWNLOAD SELURUH ARTIFAK HASIL ANALISIS
# ==========================================
artifacts = [
    'ulasan_mentah.csv',
    'ulasan_screening.csv',
    'ulasan_terproses.csv',
    'top20_unigrams_per_sentimen.csv',
    'saran_pengembangan_mytelkomsel.csv',
    'distribusi_sentimen.png',
    'barchart_top20_kata.png',
    'wordcloud_keseluruhan.png',
    'wordcloud_positif.png',
    'wordcloud_negatif.png'
]

zip_output_name = 'hasil_analisis_mytelkomsel.zip'

with zipfile.ZipFile(zip_output_name, 'w') as zipf:
    for file_path in artifacts:
        if os.path.exists(file_path):
            zipf.write(file_path)
            print(f"📦 Ditambahkan ke arsip: {file_path}")

print(f"\n🎉 Berkas arsip '{zip_output_name}' berhasil dibuat!")

# Trigger download otomatis jika dijalankan di Google Colab
try:
    from google.colab import files
    files.download(zip_output_name)
    print("⬇️ Browser mengunduh file zip secara otomatis...")
except ImportError:
    print("ℹ️ Script dijalankan di lingkungan lokal. Berkas arsip tersimpan di direktori kerja.")
